# 04 · Model version and release

Package the model selected in notebook 02 with its data, code, settings and metrics, then prove the package is complete: every file is checksum-verified, and the reloaded model reproduces the original forecasts exactly.

In [1]:
import json
from pathlib import Path

import pandas as pd

from supplysignal.release import package, verify

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN = PROJECT / "artifacts/notebook-run"
ARTIFACT = RUN / "demand_model.joblib"

## 1. Model identity

The version ID is derived from the training data, code, configuration and library versions, so the same inputs always produce the same ID.

In [2]:
lineage = json.loads(ARTIFACT.with_suffix(".manifest.json").read_text())
report = json.loads((RUN / "training_report.json").read_text())
assert report["model_version"] == lineage["model_version"]

pd.Series({
    "version": lineage["model_version"],
    "model": report["selected_model"],
    "selected by": "lowest validation MAE",
    "python": lineage["python"],
    "ledger sha256": lineage["ledger_sha256"],
    "model sha256": lineage["artifact_sha256"],
}).to_frame("value")

,value
version,demand-v1-965579aefe635327
model,lightgbm
selected by,lowest validation MAE
python,3.13.9
ledger sha256,247faf26cfa9ca13cfa275614f3a9b8bc47a3e37eaca91...
model sha256,3e1011e3c62fe7e0ee9c8fdd257058e7ca87d22bb26490...


In [3]:
pd.Series(lineage["configuration"]["selected_parameters"]).to_frame("setting")

,setting
boosting_type,gbdt
class_weight,None
colsample_bytree,1.0
importance_type,split
learning_rate,0.05
max_depth,-1
min_child_samples,20
min_child_weight,0.001
min_split_gain,0.0
n_estimators,180


## 2. Build and verify the release

`package` copies the model, ledger, training report, test predictions and source code into a versioned folder and ZIP. `verify` re-hashes every file and reloads the model to confirm it reproduces all 84 current forecasts exactly. An existing release with the same version is verified, never overwritten.

In [4]:
release = package(ARTIFACT, PROJECT / "data/processed/synthetic_lmis.csv", RUN / "training_report.json", RUN / "test_predictions.csv")
manifest = verify(release)
assert manifest["load_parity_rows"] == 84

print(f"Release: {release.relative_to(PROJECT)} (+ .zip)")
print(f"Files checksum-verified: {len(manifest['files_sha256'])}")
print(f"Forecasts reproduced exactly after reload: {manifest['load_parity_rows']}")

Release: artifacts/releases/demand-v1-965579aefe635327 (+ .zip)
Files checksum-verified: 30
Forecasts reproduced exactly after reload: 84


## 3. Record the release

`reports/model_release.json` is the short record the README and model card point to.

In [5]:
summary = {
    "model_version": manifest["model_version"],
    "status": manifest["status"],
    "path": str(release.relative_to(PROJECT)),
    "files": len(manifest["files_sha256"]),
    "load_parity_rows": manifest["load_parity_rows"],
    "selected_test_metrics": report["test_metrics"][report["selected_model"]],
}
(PROJECT / "reports/model_release.json").write_text(json.dumps(summary, indent=2) + "\n")
summary

{'model_version': 'demand-v1-965579aefe635327',
 'status': 'SIMULATION PROTOTYPE; no field validation',
 'path': 'artifacts/releases/demand-v1-965579aefe635327',
 'files': 30,
 'load_parity_rows': 84,
 'selected_test_metrics': {'n': 336,
  'mae': 2.6712,
  'wape': 0.1666,
  'bias_units': -0.3316}}

## What this means

The demand model is a single, traceable artifact: anyone can confirm which data and code produced it, and that it behaves identically after being moved. A real LGA deployment retrains on its own history and gets a new version ID through the same process.